# Ingestion Google Sheets
## Python + gspread → Delta Lake

**Objectif :** Ingérer des données directement depuis Google Sheets 
vers Delta Lake sans passer par GCS.

**Avantages vs GoogleSheetsToGCSOperator :**
- Pas de bucket GCS intermédiaire
- Contrôle total sur la transformation
- Natif Databricks pas de dépendance Airflow

**Flux :**
Google Sheets → gspread (Python) → Spark DataFrame → Delta Lake

In [0]:
%python
# Installation des librairies Google
%pip install gspread google-auth

### Authentification Google Cloud

**Prérequis :**
- Service Account GCP avec accès Google Sheets API
- Fichier JSON credentials uploadé dans le Volume

**Sécurité :** Les credentials ne sont jamais hardcodés 
stockés dans le Volume Unity Catalog.

In [0]:
%python
import gspread
from google.oauth2.service_account import Credentials
import json

# Lecture des credentials depuis le Volume Unity Catalog
# Le fichier JSON est stocké dans /Volumes/sport_metrics/raw/files/
credentials_path = "/Volumes/sport_metrics/raw/files/credentials.json"

scopes = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(credentials_path, scopes=scopes)
client = gspread.authorize(creds)

print("Authentification Google réussie")

In [0]:
# ============================================
# Configuration des 5 Google Sheets
# ============================================
sheets_config = {
    "team_players_personal_info" : "1-3xudFVhIVOHtzdBJwPd4pEkH8r-9QKF2ObLEiw_oyE",
    "team_boxscores"             : "1B22H9gjHXf-uJdoWfXrEV3np7Da4ZiCqQ4k2VP9lzbs",
    "team_training_sessions"     : "1xkUO7yzOOuOmrOPRJ8OEM6lAKgeTBsjmXiVVrlgiVdU",
    "team_games_dataset"         : "1BNsP9ZNuTn1JXmJyZyUs6gKQ3Op2v3AUAI46A7SZpIE",
    "team_players_stats"         : "1GvkwcHh60AfJr2i8pUqjr-FTpZBOZIKPQn4ykSUrhrk"
}

print(f" {len(sheets_config)} sheets configurés")

In [0]:
# ============================================
# Ingestion : Google Sheets → Delta Lake
# Pour chaque sheet :
# 1. Lecture via gspread
# 2. Conversion Pandas → Spark DataFrame
# 3. Écriture Delta Lake en mode overwrite
# ============================================

import pandas as pd
import numpy as np
import re

def parse_minutes(val):
    """Convertit MM:SS ou MM:SS:00 en minutes décimales"""
    if val is None or str(val).strip() == '':
        return None
    val = str(val).strip()
    parts = val.split(':')
    try:
        if len(parts) >= 2:
            minutes = float(parts[0])
            seconds = float(parts[1])
            return round(minutes + seconds / 60, 2)
        else:
            return round(float(val), 2)
    except:
        return None

for table_name, sheet_id in sheets_config.items():
    try:
        sheet = client.open_by_key(sheet_id)
        worksheet = sheet.get_worksheet(0)

        # FORMATTED_VALUE pour team_players_stats → garde MM:SS
        # UNFORMATTED_VALUE pour les autres → garde les nombres
        render_option = 'FORMATTED_VALUE' if table_name in ['team_players_stats', 'team_training_sessions'] else 'UNFORMATTED_VALUE'

        data = worksheet.get_all_records(
            expected_headers=[],
            value_render_option=render_option
        )

        if not data:
            print(f"{table_name} → vide")
            continue

        # Conversion Pandas
        pdf = pd.DataFrame(data)

        # Traitement spécial MIN pour team_players_stats
        if table_name == 'team_players_stats' and 'MIN' in pdf.columns:
            pdf['MIN'] = pdf['MIN'].apply(
                lambda x: parse_minutes(x) if ':' in str(x) else x
            )

        # Typage intelligent colonne par colonne
        for col in pdf.columns:
            # Remplacer les vides par NaN
            pdf[col] = pdf[col].replace('', np.nan)

            # Essayer numérique d'abord
            try:
                converted = pd.to_numeric(pdf[col], errors='raise')
                if converted.dropna().apply(lambda x: x == int(x)).all():
                    pdf[col] = converted.astype('Int64')
                else:
                    pdf[col] = converted.astype('float64')
            except:
                # Garder en string si pas numérique
                pdf[col] = pdf[col].astype(str).replace('nan', None)

        # Conversion Pandas → Spark
        sdf = spark.createDataFrame(pdf)

        # Écriture Delta Lake — remplace directement raw
        sdf.write \
           .format("delta") \
           .mode("overwrite") \
           .option("overwriteSchema", "true") \
           .saveAsTable(f"sport_metrics.raw.{table_name}")

        print(f"{table_name} → {len(data)} lignes · types détectés automatiquement")

    except Exception as e:
        print(f"{table_name} → Erreur : {e}")

In [0]:
%sql
SELECT MIN(min), MAX(min), AVG(min)
FROM sport_metrics.raw.team_players_stats;